# Inference: U-Net Burn Detection

Run trained model on test set to generate predictions for evaluation.

## Setup

In [ ]:
import os
if not os.path.exists('RETINNA'):
    !git clone https://github.com/scerruti/RETINNA.git

import sys
sys.path.insert(0, '/content/RETINNA' if '/content' in os.getcwd() else 'RETINNA')

In [ ]:
%pip install -q torchgeo torch torchvision matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.unet import UNet
from src.dataset import get_dataloaders
from src.device_utils import get_device, move_to_device

device = get_device()

## Load Data and Model

In [ ]:
# Setup Google Drive caching if on Colab
root = None
try:
    from src.colab_utils import setup_cabuaur_cached
    root = setup_cabuaur_cached()
    print("✓ Google Drive caching enabled")
except (ImportError, RuntimeError):
    print("Using default TorchGeo cache")

# Load dataloaders (test set only)
print("\nLoading test set...")
dataloaders = get_dataloaders(
    batch_size=4,
    num_workers=0,
    normalize=True,
    root=root
)

test_loader = dataloaders['test']
print(f"Test samples: {len(dataloaders['datasets']['test'])}")


In [ ]:
# Load trained model
checkpoint_path = Path('checkpoints_notebook/best_model.pth')

# If checkpoint doesn't exist locally, copy from Google Drive
if not checkpoint_path.exists():
    drive_checkpoint = Path('/content/drive/MyDrive/RETINNA_checkpoints/best_model.pth')
    if drive_checkpoint.exists():
        print(f"Checkpoint not found locally, copying from Google Drive...")
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(drive_checkpoint, checkpoint_path)
        print(f"✓ Copied {drive_checkpoint} → {checkpoint_path}")
    else:
        print(f"Error: Checkpoint not found at {checkpoint_path} or {drive_checkpoint}")
        print("Make sure to run 03_training.ipynb first and save the model.")

if checkpoint_path.exists():
    print(f"Loading checkpoint: {checkpoint_path}")
    model = UNet(in_channels=24, out_channels=2).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    print(f"✓ Model loaded: {model.get_parameter_count():,} parameters")

## Run Inference

In [ ]:
# Store predictions and ground truth
all_predictions = []
all_targets = []
all_images = []

print("Running inference on test set...")
with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        batch = move_to_device(batch, device)
        images = batch['image']
        masks = batch['mask']

        # Flatten timesteps
        B, T, C, H, W = images.shape
        images_flat = images.view(B, T * C, H, W)

        # Forward pass
        outputs = model(images_flat)  # [B, 2, H, W]
        probs = torch.softmax(outputs, dim=1)  # Probabilities
        burned_prob = probs[:, 1]  # Probability of burned class [B, H, W]

        # Store on CPU
        all_predictions.append(burned_prob.cpu())
        all_targets.append(masks.squeeze(1).cpu())
        all_images.append(images[:, 1].cpu())  # Store POST-FIRE image (timestep 1, not pre-fire!)

        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {batch_idx + 1}/{len(test_loader)} batches")

# Concatenate all predictions
predictions = torch.cat(all_predictions, dim=0)
targets = torch.cat(all_targets, dim=0)
images = torch.cat(all_images, dim=0)

print(f"\n✓ Inference complete")
print(f"  Predictions shape: {predictions.shape}")
print(f"  Targets shape: {targets.shape}")
print(f"  Images shape: {images.shape}")

# Save predictions locally
output_dir = Path('inference_results')
output_dir.mkdir(exist_ok=True)
local_path = output_dir / 'predictions.pt'
torch.save({'predictions': predictions, 'targets': targets, 'images': images}, local_path)
print(f"\n✓ Saved predictions to {local_path}")

# Backup to Google Drive
try:
    drive_output_dir = Path('/content/drive/MyDrive/RETINNA_checkpoints')
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    drive_path = drive_output_dir / 'predictions.pt'
    import shutil
    shutil.copy(local_path, drive_path)
    print(f"✓ Backed up to Google Drive: {drive_path}")
except Exception as e:
    print(f"⚠ Could not backup to Drive: {e}")

## Sample Predictions

In [ ]:
# Show a few sample predictions
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample Predictions: Post-Fire Image | Ground Truth | Prediction | Binary (T=0.5)', fontsize=12)

for i in range(3):
    # Post-fire image (SWIR band for visualization)
    ax = axes[i, 0]
    img = images[i, 10].numpy()  # SWIR band (index 10) from POST-FIRE timestep
    ax.imshow(img, cmap='gray')
    ax.set_title('Post-Fire')
    ax.axis('off')

    # Ground truth
    ax = axes[i, 1]
    ax.imshow(targets[i].numpy(), cmap='RdYlGn_r', vmin=0, vmax=1)
    ax.set_title('Ground Truth')
    ax.axis('off')

    # Prediction probability
    ax = axes[i, 2]
    ax.imshow(predictions[i].numpy(), cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title('Prediction (Prob)')
    ax.axis('off')

    # Binary prediction (threshold 0.5)
    ax = axes[i, 3]
    pred_binary = (predictions[i] > 0.5).float()
    ax.imshow(pred_binary.numpy(), cmap='RdYlGn_r', vmin=0, vmax=1)
    ax.set_title('Binary (T=0.5)')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\nSample predictions saved. Use 05_analysis.ipynb for detailed metrics.")

In [ ]:
# Analyze performance distribution across test set
threshold = 0.1
import torch

sample_recalls = []
sample_precisions = []
sample_sizes = []

print("Analyzing per-sample performance...\n")
print(f"{'Sample':<8} {'Burned%':<10} {'Recall%':<10} {'Precision%':<12} {'Status':<15}")
print("-" * 60)

for i in range(min(20, len(predictions))):  # Analyze first 20 samples
    pred_binary = (predictions[i] > threshold).numpy()
    target_binary = targets[i].numpy()
    
    tp = ((pred_binary == 1) & (target_binary == 1)).sum()
    fp = ((pred_binary == 1) & (target_binary == 0)).sum()
    fn = ((pred_binary == 0) & (target_binary == 1)).sum()
    
    burned_pct = target_binary.sum() / target_binary.size * 100
    recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    
    sample_recalls.append(recall)
    sample_precisions.append(precision)
    sample_sizes.append(target_binary.sum())
    
    status = "✓ Good" if recall > 50 else "⚠ Poor" if recall > 20 else "✗ Failed"
    print(f"{i:<8} {burned_pct:<10.1f} {recall:<10.1f} {precision:<12.1f} {status:<15}")

print("-" * 60)
print(f"{'Mean':<8} {np.mean([targets[i].numpy().sum()/targets[i].numpy().size*100 for i in range(min(20, len(predictions)))]):<10.1f} {np.mean(sample_recalls):<10.1f} {np.mean(sample_precisions):<12.1f}")
print(f"{'Std':<8} {'':<10} {np.std(sample_recalls):<10.1f} {np.std(sample_precisions):<12.1f}")
print(f"{'Min':<8} {'':<10} {np.min(sample_recalls):<10.1f} {np.min(sample_precisions):<12.1f}")
print(f"{'Max':<8} {'':<10} {np.max(sample_recalls):<10.1f} {np.max(sample_precisions):<12.1f}")

print(f"\n⚠️  High Variance Alert: Std Dev = {np.std(sample_recalls):.1f}%")
print(f"This indicates model performs very differently across tiles.")


In [ ]:
# Error Analysis: Show True Positives, False Positives, False Negatives
# Color coding: Green=TP, Red=FN, Orange=FP, Gray=TN

threshold = 0.1  # Use T=0.1 which gives best recall (71.1%)

fig, axes = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle(f'Error Analysis @ Threshold={threshold} | Left: Error Map | Right: Prediction Heatmap', fontsize=13, fontweight='bold')

for row in range(3):
    # Error map
    ax = axes[row, 0]
    pred_binary = (predictions[row] > threshold).numpy()
    target_binary = targets[row].numpy()
    
    # Create error map: TP, FP, FN, TN
    error_map = np.zeros((512, 512, 3))
    tp = (pred_binary == 1) & (target_binary == 1)
    fp = (pred_binary == 1) & (target_binary == 0)
    fn = (pred_binary == 0) & (target_binary == 1)
    tn = (pred_binary == 0) & (target_binary == 0)
    
    error_map[tp] = [0, 1, 0]      # TP = Green
    error_map[fp] = [1, 0.6, 0]    # FP = Orange
    error_map[fn] = [1, 0, 0]      # FN = Red
    error_map[tn] = [0.7, 0.7, 0.7]  # TN = Gray (dimmed)
    
    ax.imshow(error_map)
    tp_count = tp.sum()
    fp_count = fp.sum()
    fn_count = fn.sum()
    recall = tp_count / (tp_count + fn_count) * 100 if (tp_count + fn_count) > 0 else 0
    precision = tp_count / (tp_count + fp_count) * 100 if (tp_count + fp_count) > 0 else 0
    
    ax.set_title(f'Error Map | TP={tp_count} FP={fp_count} FN={fn_count}\nRecall={recall:.1f}% Prec={precision:.1f}%', fontsize=10)
    ax.axis('off')
    
    # Prediction heatmap (actual range 0-0.35)
    ax = axes[row, 1]
    im = ax.imshow(predictions[row].numpy(), cmap='YlOrRd', vmin=0, vmax=0.35)
    ax.set_title('Prediction Heatmap (Range: 0-0.35)', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('inference_results/error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"✓ Error analysis visualization saved: inference_results/error_analysis.png")
print(f"\nLegend:")
print(f"  Green  = True Positive (correctly predicted burn)")
print(f"  Red    = False Negative (missed burn)")
print(f"  Orange = False Positive (incorrectly predicted burn)")
print(f"  Gray   = True Negative (correctly predicted unburned)")
print(f"\nThreshold: {threshold} (gives ~71% recall)")